# 08. Validation
**목적**: 모델 결과의 타당성을 다층적으로 검증한다.

1. **Ablation Study** — feature group 제거 시 고위험 설비 식별력 변화
2. **Sensitivity Analysis** — 가중치 시나리오별 Top-K overlap
3. **External Validation** — 산불발생통계 / 공식 산불위험지수와 방향성 비교 (허용 시)
4. **분포 기반 합리성 검토** — 건조·강풍일에 위험도 상승 확인

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

from config import DATA_PROCESSED, OUT_FIGURES, OUT_TABLES, WEIGHTS, TOP_K_PCTS

with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FLAGS = cmap['flags']
FC = cmap['facility']
FID_COL = FC['facility_id']

df = pd.read_parquet(DATA_PROCESSED / 'risk_scores.parquet')
print(f'데이터: {df.shape}')

## 1. Ablation Study

In [ ]:
# 각 모델이 Top-10% 설비에서 얼마나 다른지 비교
def topk_set(df, score_col, pct=0.10):
    k = max(1, int(df[FID_COL].nunique() * pct))
    return set(df.groupby(FID_COL)[score_col].mean().nlargest(k).index)


# 4개 ablation 모델 (weather_hazard만 있는 경우 ~ 전체)
df['score_M1'] = df['weather_hazard'].fillna(50)   # Weather only

df['score_M2'] = (
    0.60 * df['weather_hazard'].fillna(50) +
    0.40 * df['facility_exposure'].fillna(50)
)  # Weather + Facility

df['score_M3'] = (
    0.45 * df['weather_hazard'].fillna(50) +
    0.30 * df['spatial_exposure'].fillna(50) +
    0.25 * df['facility_exposure'].fillna(50)
)  # Weather + Facility + Spatial

df['score_M4'] = df['final_risk']   # Full model

ablation_models = ['score_M1', 'score_M2', 'score_M3', 'score_M4']
model_labels    = ['M1: Weather', 'M2: +Facility', 'M3: +Spatial', 'M4: Full']

top10_sets = {m: topk_set(df, m, pct=0.10) for m in ablation_models}
base_set = top10_sets['score_M4']

ablation_rows = []
for m, label in zip(ablation_models, model_labels):
    s = top10_sets[m]
    overlap = len(s & base_set) / len(base_set) * 100
    ablation_rows.append({'Model': label, 'Top10% Overlap with M4 (%)': round(overlap, 1)})

df_ablation = pd.DataFrame(ablation_rows)
print('=== Ablation Study ===')
print(df_ablation.to_string(index=False))
df_ablation.to_csv(OUT_TABLES / 'ablation_study.csv', index=False)

In [ ]:
# Ablation bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#aec6e8', '#7bafd4', '#3a7ebf', '#1a4f8a']
bars = ax.barh(df_ablation['Model'], df_ablation['Top10% Overlap with M4 (%)'],
               color=colors, edgecolor='white')
for bar, val in zip(bars, df_ablation['Top10% Overlap with M4 (%)']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=11)
ax.set_xlabel('Full 모델과의 Top-10% 설비 일치율 (%)', fontsize=11)
ax.set_title('Ablation Study — Feature Group별 기여도', fontsize=13)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('ablation_study.png 저장')

## 2. Sensitivity Analysis (가중치 시나리오)

In [ ]:
scenario_cols = [f'final_risk_{s}' for s in WEIGHTS.keys()]
scenario_cols = [c for c in scenario_cols if c in df.columns]

base_top10 = topk_set(df, 'final_risk', pct=0.10)

sens_rows = []
for sc in scenario_cols:
    name = sc.replace('final_risk_', '')
    s = topk_set(df, sc, pct=0.10)
    overlap = len(s & base_top10) / len(base_top10) * 100
    sens_rows.append({'Scenario': name, 'Top10% Overlap (%)': round(overlap, 1)})

df_sens = pd.DataFrame(sens_rows)
print('=== Sensitivity Analysis ===')
print(df_sens.to_string(index=False))
df_sens.to_csv(OUT_TABLES / 'sensitivity_analysis.csv', index=False)

min_overlap = df_sens['Top10% Overlap (%)'].min()
print(f'\n최소 overlap: {min_overlap:.0f}% → 가중치 변화에도 Top-10% 설비 {min_overlap:.0f}% 이상 유지')

## 3. 분포 기반 합리성 검토

In [ ]:
# 건조·강풍일 vs 일반일 위험도 비교
df['is_dry_wind'] = (
    (df['weather_hazard'] >= df['weather_hazard'].quantile(0.75))
).astype(int)

risk_by_condition = df.groupby('is_dry_wind')['final_risk'].describe()
print('=== 고위험 기상 조건별 최종 위험도 ===')
print(risk_by_condition.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
groups = ['일반 기상일 (하위 75%)', '고위험 기상일 (상위 25%)']
means  = [df[df['is_dry_wind']==0]['final_risk'].mean(),
           df[df['is_dry_wind']==1]['final_risk'].mean()]
stds   = [df[df['is_dry_wind']==0]['final_risk'].std(),
           df[df['is_dry_wind']==1]['final_risk'].std()]

ax.bar(groups, means, color=['#aec6e8', '#d62728'], yerr=stds, capsize=5, edgecolor='white')
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.5, f'{m:.1f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('평균 최종 위험도 점수', fontsize=11)
ax.set_title('기상 조건별 전력설비 평균 위험도 비교', fontsize=13)
ax.set_ylim(0, max(means) * 1.3)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'risk_by_weather_condition.png', dpi=150, bbox_inches='tight')
plt.show()
print('risk_by_weather_condition.png 저장')

## 4. External Validation (외부 데이터 허용 시)

In [ ]:
from config import DATA_EXTERNAL
FIRE_FILE = DATA_EXTERNAL / 'fire_history.csv'

if FLAGS['allow_external_spatial'] and FIRE_FILE.exists():
    import geopandas as gpd
    from config import CRS_PROJ

    df_fire = pd.read_csv(FIRE_FILE, encoding='utf-8-sig')
    gdf_fire = gpd.GeoDataFrame(
        df_fire,
        geometry=gpd.points_from_xy(df_fire['fire_lon'], df_fire['fire_lat']),
        crs='EPSG:4326'
    ).to_crs(CRS_PROJ)

    gdf_fac = gpd.read_file(DATA_PROCESSED / 'facility_proj.gpkg')
    latest_risk = df.groupby(FID_COL)['final_risk'].mean().reset_index()
    gdf_fac = gdf_fac.merge(latest_risk, on=FID_COL, how='left')

    # Top 10% vs Bottom 90% 설비 주변 산불 발생 밀도 비교
    q90 = gdf_fac['final_risk'].quantile(0.90)
    gdf_high_risk = gdf_fac[gdf_fac['final_risk'] >= q90].buffer(5000)   # 5km
    gdf_low_risk  = gdf_fac[gdf_fac['final_risk'] <  q90].buffer(5000)

    fire_in_high = gdf_fire.within(gdf_high_risk.unary_union).sum()
    fire_in_low  = gdf_fire.within(gdf_low_risk.unary_union).sum()

    high_count = (gdf_fac['final_risk'] >= q90).sum()
    low_count  = (gdf_fac['final_risk'] <  q90).sum()

    print('=== External Validation — 산불 발생 밀도 비교 ===')
    print(f'Top 10% 고위험 설비 ({high_count}개) 주변 5km 내 산불: {fire_in_high}건 '
          f'(건당 설비 수: {fire_in_high/max(1,high_count):.2f})')
    print(f'나머지 설비 ({low_count}개) 주변 5km 내 산불: {fire_in_low}건 '
          f'(건당 설비 수: {fire_in_low/max(1,low_count):.2f})')
else:
    print('외부 검증 데이터 없음 — 건너뜀')

print('\n다음 단계: 09_mapping_outputs.ipynb')